# 09 — Figures + Tables for the Paper

**Day 6.** Produce all paper-ready outputs:

* Figure 1 — country-year exposure trajectories with vintage break
* Figure 2 — caterpillar plot of country random intercepts (M3a)
* Figure 3 — conditional effects of within × education (M5)
* Figure 4 — variance components M0 → M6
* Table 1 — descriptive statistics
* Table 2 — master regression M0 → M6
* Table 3 — Mundlak Wald + robustness summary

All figures saved as `.pdf` to `paper/figures/`; all tables as `.tex` to `paper/tables/`.

In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt
import pandas as pd

warnings.filterwarnings("ignore")

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != "MLA" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

FIGURES_DIR = REPO_ROOT / "paper" / "figures"
TABLES_DIR  = REPO_ROOT / "paper" / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

INTERIM_DIR = REPO_ROOT / "data" / "interim"
ANALYSIS_DIR = REPO_ROOT / "data" / "analysis"

from src.mla.models import (  # noqa: E402
    add_country_year_key, build_trust_composite, fit_3level,
)
from src.mla.plotting import (  # noqa: E402
    caterpillar_country_intercepts,
    conditional_within_x_education,
    country_year_exposure_trajectories,
    variance_components_bar,
)
REPO_ROOT

PosixPath('/Users/karlalucic/Code/coursework/KUL/2sem/MLA')

## 1. Reproduce the analysis frame (same prep as notebooks 06/07/08)

In [2]:
def recode_sentinels(s, lo, hi):
    return s.where(s.between(lo, hi))

df = pd.read_parquet(ANALYSIS_DIR / "analysis.parquet")
df = build_trust_composite(df)
df = add_country_year_key(df)
df["agea"]    = recode_sentinels(df["agea"], 14, 110)
df["gndr"]    = recode_sentinels(df["gndr"], 1, 2)
df["eisced"]  = recode_sentinels(df["eisced"], 0, 7)
df["hinctnta"] = recode_sentinels(df["hinctnta"], 1, 10)
df["mnactic"] = recode_sentinels(df["mnactic"], 1, 9)
df["domicil"] = recode_sentinels(df["domicil"], 1, 5)
df["agea_c"] = df["agea"] - 45
df["agea_c_sq"] = df["agea_c"] ** 2
df["female"] = (df["gndr"] == 2).astype("float64")
for _c in ("essround", "isco08", "year"):
    if str(df[_c].dtype).startswith("Int"):
        df[_c] = df[_c].astype("float64")
df["genai_z"] = (df["genai_i"] - df["genai_i"].mean()) / df["genai_i"].std()
df["eisced_c"] = df["eisced"] - 4
REQUIRED = [
    "trust", "genai_i", "genai_z", "eisced", "eisced_c", "agea_c", "female",
    "mnactic", "domicil", "hinctnta", "gdp_growth", "unemp_rate",
    "hicp_inflation", "exposure_ct", "exposure_ct_within", "exposure_ct_between",
]
df_fit = df.dropna(subset=REQUIRED).copy()
g = df_fit.groupby("cntry", observed=True)["genai_z"].transform("mean")
df_fit["genai_z_gmc"] = df_fit["genai_z"] - g
print(f"analysis sample: {len(df_fit):,}, {df_fit.cntry.nunique()} countries")

analysis sample: 165,969, 30 countries


## 2. Figure 1 — country-year exposure trajectories

In [3]:
cy = pd.read_parquet(INTERIM_DIR / "country_year_exposure.parquet")
fig1 = country_year_exposure_trajectories(cy)
fig1.savefig(FIGURES_DIR / "fig1_exposure_trajectories.pdf", bbox_inches="tight")
fig1.savefig(FIGURES_DIR / "fig1_exposure_trajectories.png", dpi=160, bbox_inches="tight")
plt.close(fig1)
print("wrote figures: fig1_exposure_trajectories.{pdf,png}")

wrote figures: fig1_exposure_trajectories.{pdf,png}


## 3. Figure 2 — caterpillar of country random intercepts (M3a)

In [4]:
M3A_FORMULA = (
    "trust ~ genai_z + C(eisced) + agea_c + agea_c_sq + female "
    "+ C(mnactic) + C(domicil) + hinctnta "
    "+ gdp_growth + unemp_rate + hicp_inflation + C(essround) "
    "+ exposure_ct_within + exposure_ct_between"
)
res3a = fit_3level(M3A_FORMULA, df_fit)
fig2 = caterpillar_country_intercepts(res3a)
fig2.savefig(FIGURES_DIR / "fig2_caterpillar.pdf", bbox_inches="tight")
fig2.savefig(FIGURES_DIR / "fig2_caterpillar.png", dpi=160, bbox_inches="tight")
plt.close(fig2)
print("wrote figures: fig2_caterpillar.{pdf,png}")

wrote figures: fig2_caterpillar.{pdf,png}


/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


## 4. Figure 3 — conditional within × education (M5)

In [5]:
M5_FORMULA = M3A_FORMULA + " + exposure_ct_within:eisced_c"
res5 = fit_3level(M5_FORMULA, df_fit, re_formula="~1 + genai_z_gmc")
fig3 = conditional_within_x_education(res5)
fig3.savefig(FIGURES_DIR / "fig3_conditional_within_x_education.pdf", bbox_inches="tight")
fig3.savefig(FIGURES_DIR / "fig3_conditional_within_x_education.png", dpi=160, bbox_inches="tight")
plt.close(fig3)
print("wrote figures: fig3_conditional_within_x_education.{pdf,png}")

/Users/karlalucic/Code/coursework/KUL/2sem/MLA/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


wrote figures: fig3_conditional_within_x_education.{pdf,png}


## 5. Figure 4 — variance components M0 → M6

In [6]:
tbl_m0_m6 = pd.read_parquet(INTERIM_DIR / "master_table_m0_m6.parquet")
fig4 = variance_components_bar(tbl_m0_m6)
fig4.savefig(FIGURES_DIR / "fig4_variance_components.pdf", bbox_inches="tight")
fig4.savefig(FIGURES_DIR / "fig4_variance_components.png", dpi=160, bbox_inches="tight")
plt.close(fig4)
print("wrote figures: fig4_variance_components.{pdf,png}")

wrote figures: fig4_variance_components.{pdf,png}


## 6. Table 1 — descriptive statistics

In [7]:
desc_vars = ["trust", "genai_z", "agea", "female", "hinctnta",
             "gdp_growth", "unemp_rate", "hicp_inflation",
             "exposure_ct", "exposure_ct_within", "exposure_ct_between"]
labels = {
    "trust": "Trust composite (z)",
    "genai_z": "Individual ILO–NASK GenAI exposure (z)",
    "agea": "Age (years)",
    "female": "Female (0/1)",
    "hinctnta": "Household income decile (1–10)",
    "gdp_growth": "Real GDP growth (\\%)",
    "unemp_rate": "Unemployment rate (\\%)",
    "hicp_inflation": "HICP inflation (\\%)",
    "exposure_ct": r"$\bar E_{ct}$ (country-year exposure)",
    "exposure_ct_within": r"$E_{ct} - \bar E_c$ (within)",
    "exposure_ct_between": r"$\bar E_c$ (between)",
}
desc = df_fit[desc_vars].describe().T
desc.index = [labels[v] for v in desc.index]
desc = desc[["count", "mean", "std", "min", "50%", "max"]]
desc.columns = ["N", "Mean", "SD", "Min", "Median", "Max"]
desc["N"] = desc["N"].astype(int)
tex = desc.round(2).to_latex(
    escape=False, column_format="lrrrrrr",
    caption="Descriptive statistics, M3a analysis sample (N = 165{,}969).",
    label="tab:descriptives",
)
(TABLES_DIR / "tab1_descriptives.tex").write_text(tex)
print("wrote paper/tables/tab1_descriptives.tex")
desc.round(2)

wrote paper/tables/tab1_descriptives.tex


,N,Mean,SD,Min,Median,Max
Trust composite (z),165969,0.07,0.85,-2.02,0.15,2.11
Individual ILO–NASK GenAI exposure (z),165969,-0.01,0.99,-1.41,-0.13,2.79
Age (years),165969,51.67,17.36,14.00,52.00,104.00
Female (0/1),165969,0.52,0.50,0.00,1.00,1.00
Household income decile (1–10),165969,5.41,2.73,1.00,5.00,10.00
Real GDP growth (\%),165969,2.41,2.84,-4.10,2.10,16.30
Unemployment rate (\%),165969,7.58,3.88,2.20,6.80,24.80
HICP inflation (\%),165969,2.64,2.74,-0.70,2.10,17.00
$\bar E_{ct}$ (country-year exposure),165969,0.31,0.02,0.26,0.31,0.35
$E_{ct} - \bar E_c$ (within),165969,-0.00,0.01,-0.02,0.00,0.02


## 7. Table 2 — master regression M0 → M6

In [8]:
# Build a compact paper-style table: one column per model, rows for the
# coefficients of interest + variance components.
# Use \makecell[r]{...\\...} (not \newline) for stacked beta/SE within an r-column.

PRETTY_TERM = {
    "genai_z":                          r"Individual GenAI exposure ($z$)",
    "exposure_ct_within":               r"$E_{ct}-\bar E_c$ (within)",
    "exposure_ct_between":              r"$\bar E_c$ (between)",
    "exposure_ct_within:eisced_c":      r"Within $\times$ ISCED",
    "epl_c_centred":                    r"EPL (centred)",
    "exposure_ct_within:epl_c_centred": r"Within $\times$ EPL",
}

tbl = tbl_m0_m6.copy()
rows = []
for coef, pretty in PRETTY_TERM.items():
    row = {"term": pretty}
    for _, r in tbl.iterrows():
        beta = r.get(f"{coef}__beta", float("nan"))
        se = r.get(f"{coef}__se", float("nan"))
        if pd.isna(beta):
            row[r["model"]] = ""
        else:
            z = beta / se if se else float("nan")
            sig = "^{*}" if abs(z) > 1.96 else ""
            row[r["model"]] = f"\\makecell[r]{{${beta:+.3f}{sig}$\\\\({se:.3f})}}"
    rows.append(row)
rows.append({"term": r"$\sigma^2_{u_0}$ (L3)", **{r["model"]: f"{r['sigma_u0_sq']:.3f}" for _, r in tbl.iterrows()}})
rows.append({"term": r"$\sigma^2_{v_0}$ (L2)", **{r["model"]: f"{r['sigma_v0_sq']:.3f}" for _, r in tbl.iterrows()}})
rows.append({"term": r"$\sigma^2_e$  (L1)",   **{r["model"]: f"{r['sigma_e_sq']:.3f}"  for _, r in tbl.iterrows()}})
rows.append({"term": "ICC L3",                  **{r["model"]: f"{r['icc_l3']:.3f}"      for _, r in tbl.iterrows()}})
rows.append({"term": "VPC L2",                  **{r["model"]: f"{r['vpc_l2']:.3f}"      for _, r in tbl.iterrows()}})
rows.append({"term": "N",                       **{r["model"]: f"{int(r['n_obs']):,}"    for _, r in tbl.iterrows()}})
rows.append({"term": "countries (L3)",          **{r["model"]: f"{int(r['n_groups_l3'])}" for _, r in tbl.iterrows()}})
wide = pd.DataFrame(rows).set_index("term")
wide.index.name = None  # avoid the extra blank row pandas inserts
tex2 = wide.to_latex(
    escape=False, column_format="l" + "r" * len(tbl),
    caption="Three-level multilevel models of the institutional-trust composite, M0 to M6. "
            "Coefficients (standard errors). $^{*}\\,p<0.05$.",
    label="tab:models",
)
(TABLES_DIR / "tab2_models_m0_m6.tex").write_text(tex2)
print("wrote paper/tables/tab2_models_m0_m6.tex")
wide


wrote paper/tables/tab2_models_m0_m6.tex


,M0,M1,M2,M3a,M3b,M4,M5,M6
Individual GenAI exposure ($z$),,\makecell[r]{$+0.015^{*}$\\(0.002)},\makecell[r]{$+0.015^{*}$\\(0.002)},\makecell[r]{$+0.015^{*}$\\(0.002)},\makecell[r]{$+0.015^{*}$\\(0.002)},\makecell[r]{$+0.008$\\(0.021)},\makecell[r]{$+0.007$\\(0.008)},\makecell[r]{$+0.017$\\(0.016)}
$E_{ct}-\bar E_c$ (within),,,,\makecell[r]{$+0.024$\\(1.123)},\makecell[r]{$+1.459$\\(1.118)},\makecell[r]{$+0.039$\\(1.044)},\makecell[r]{$+0.203$\\(1.122)},\makecell[r]{$+0.698$\\(1.318)}
$\bar E_c$ (between),,,,\makecell[r]{$+11.239^{*}$\\(3.427)},\makecell[r]{$+11.168^{*}$\\(3.451)},\makecell[r]{$+11.180^{*}$\\(2.373)},\makecell[r]{$+10.488^{*}$\\(3.026)},\makecell[r]{$+7.542$\\(4.003)}
Within $\times$ ISCED,,,,,,,\makecell[r]{$-0.142$\\(0.130)},\makecell[r]{$-0.318^{*}$\\(0.141)}
EPL (centred),,,,,,,,\makecell[r]{$-0.107$\\(0.097)}
Within $\times$ EPL,,,,,,,,\makecell[r]{$+1.637$\\(1.442)}
$\sigma^2_{u_0}$ (L3),0.179,0.167,0.156,0.117,0.118,0.055,0.118,0.077
$\sigma^2_{v_0}$ (L2),0.018,0.015,0.007,0.007,0.009,0.006,0.007,0.008
$\sigma^2_e$ (L1),0.539,0.511,0.511,0.511,0.511,0.510,0.510,0.496
ICC L3,0.243,0.241,0.232,0.183,0.185,0.097,0.185,0.132


## 8. Table 3 — Mundlak Wald + robustness summary

In [9]:
wald = pd.read_parquet(INTERIM_DIR / "mundlak_wald_tests.parquet")
robust = pd.read_csv(INTERIM_DIR / "robustness_summary.csv")

# Tab 3a: format floats to 3 decimals (otherwise pandas defaults to 6 trailing zeros)
wald_disp = wald.copy()
wald_disp.columns = ["Model", r"$\hat\gamma_W$", r"$\hat\gamma_B$",
                     r"Diff", r"SE(diff)", r"$\chi^2$", "df", r"$p$"]
tex3a = wald_disp.to_latex(
    index=False, escape=False, float_format="%.3f",
    caption=r"Mundlak Wald test of $\gamma_W = \gamma_B$ across the M3a--M5 specifications.",
    label="tab:mundlak",
)

# Tab 3b: rename columns + math-mode-escape any bare Greek in the check labels
robust_disp = robust.copy()
robust_disp.columns = ["Check", r"$\hat\gamma_W$", r"$\hat\gamma_B$", r"Wald $p$"]
# Replace bare Unicode Greek/non-ASCII in check labels with LaTeX math-mode equivalents
robust_disp["Check"] = (
    robust_disp["Check"]
    .str.replace("γ_B", r"$\gamma_B$", regex=False)
    .str.replace("γ_W", r"$\gamma_W$", regex=False)
    .str.replace("β", r"$\beta$", regex=False)
    .str.replace("exposure_ct", r"\texttt{exposure\_ct}", regex=False)
)
tex3b = robust_disp.to_latex(
    index=False, escape=False,
    caption="Robustness battery for the M3a within-between specification.",
    label="tab:robustness",
)
(TABLES_DIR / "tab3a_mundlak.tex").write_text(tex3a)
(TABLES_DIR / "tab3b_robustness.tex").write_text(tex3b)
print("wrote paper/tables/tab3a_mundlak.tex, tab3b_robustness.tex")
print()
print("Mundlak Wald:")
print(wald.round(4))
print()
print("Robustness:")
print(robust)


wrote paper/tables/tab3a_mundlak.tex, tab3b_robustness.tex

Mundlak Wald:
  model  gamma_w  gamma_b     diff  se_diff     chi2   df  pvalue
0   M3a   0.0235  11.2390 -11.2155   3.6077   9.6643  1.0  0.0019
1   M3b   1.4591  11.1681  -9.7090   3.6266   7.1674  1.0  0.0074
2    M4   0.0393  11.1804 -11.1411   2.5961  18.4162  1.0  0.0000
3    M5   0.2035  10.4884 -10.2849   3.2335  10.1171  1.0  0.0015

Robustness:
                                           check          gamma_w  \
0                               R5 trstprl alone           1.3826   
1                               R5 trstlgl alone          -1.8302   
2                                R5 stfdem alone           0.5688   
3            R6 country drop range (γ_B min/max)  [-0.348, 0.358]   
4                                   R12 drop R10          -0.8695   
5                             R-V vintage-static          -0.2357   
6                               R4 leave-one-out           0.0633   
7  R7 PanelOLS β on exposure_ct

## 9. Verify all paper outputs landed

In [10]:
for d in (FIGURES_DIR, TABLES_DIR):
    print(f"--- {d.relative_to(REPO_ROOT)}")
    for p in sorted(d.iterdir()):
        print(f"  {p.name}  ({p.stat().st_size/1e3:.1f} KB)")

--- paper/figures
  .gitkeep  (0.0 KB)
  fig1_exposure_trajectories.pdf  (32.0 KB)
  fig1_exposure_trajectories.png  (147.8 KB)
  fig2_caterpillar.pdf  (19.7 KB)
  fig2_caterpillar.png  (52.8 KB)
  fig3_conditional_within_x_education.pdf  (19.7 KB)
  fig3_conditional_within_x_education.png  (106.8 KB)
  fig4_variance_components.pdf  (14.2 KB)
  fig4_variance_components.png  (34.3 KB)
--- paper/tables
  .gitkeep  (0.0 KB)
  tab1_descriptives.tex  (1.3 KB)
  tab2_models_m0_m6.tex  (1.8 KB)
  tab3a_mundlak.tex  (0.6 KB)
  tab3b_robustness.tex  (0.7 KB)
